In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Product Detail

In [ ]:
# Import data
detail = pd.read_csv('../data/raw/products_detail.csv')

In [ ]:
# Check the shape of the data
detail.shape

In [ ]:
# Check the data types and missing values
detail.info()

In [ ]:
# Check the number of missing values in each column
detail.isnull().sum().sort_values(ascending=False)

Các cột có 100% missing (512 dòng): ai_review_summary, short_name, is_genuine, order_count → Loại khỏi processed dataset.  

In [ ]:
# Check number of unique values in each column
detail.nunique()

In [ ]:
for col in detail.columns:
    if detail[col].nunique() == 1:
        print(f"Column '{col}' has only one unique value: {detail[col].unique()[0]}")

Các cột này không mang thông tin phân biệt giữa các sản phẩm, do đó được loại khỏi processed dataset.

In [ ]:
detail.duplicated().sum()

In [ ]:
detail.head(10)

# Aspect-Based Sentiment Analysis

## Version 1

In [ ]:
absa_v1 = pd.read_json('../data/absa/v1/full_dataset.json')

In [ ]:
absa_v1.shape

In [ ]:
absa_v1.info()

In [ ]:
absa_v1.isnull().sum()

Dữ liệu không có giá trị thiếu.

In [ ]:
absa_v1["review_length"] = absa_v1["text"].astype(str).apply(lambda x: len(x.split()))
absa_v1["review_length"].describe()

Dữ liệu về review_length cho thấy đa số bài đánh giá có độ dài ngắn, tập trung quanh 11 từ. Phân vị 25% là 5 từ và 75% là 22 từ, nghĩa là phần lớn đánh giá nằm trong khoảng từ 5 đến 22 từ. Tuy nhiên, vẫn có một số ngoại lệ rất dài, với giá trị tối đa lên đến 334 từ, làm cho độ lệch chuẩn khá lớn (19.81).

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(absa_v1["review_length"], bins=50)
plt.title("Review Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

Biểu đồ cho thấy phần lớn đánh giá có độ dài rất ngắn, tập trung ở khoảng 0–10 từ. Tần suất giảm nhanh khi số từ tăng, và chỉ có rất ít đánh giá vượt quá 100 từ. Phân bố lệch phải rõ rệt, phản ánh xu hướng người dùng viết đánh giá ngắn gọn, hiếm khi viết dài. Điều này phù hợp với thống kê mô tả trước đó và gợi ý rằng phần lớn nội dung đánh giá có thể mang tính khái quát hoặc cảm tính hơn là chi tiết.

In [ ]:
aspects = [
    "description",
    "quality",
    "packaging",
    "delivery",
    "service",
    "price"
]

for aspect in aspects:
    print(absa_v1[aspect].value_counts())

Description: đa số trung lập, ít cảm xúc rõ rệt.  
Quality: phần lớn tích cực, cho thấy sự hài lòng về chất lượng.  
Packaging: chủ yếu trung lập, bao bì không phải yếu tố nổi bật.  
Delivery: nhiều đánh giá tích cực, dịch vụ giao hàng nhìn chung tốt.  
Service: đa số trung lập, trải nghiệm dịch vụ không gây ấn tượng mạnh.  
Price: phần lớn trung lập, giá cả không phải yếu tố chính trong đánh giá.

Tổng quan: người dùng tập trung khen ngợi chất lượng và giao hàng, còn các khía cạnh khác thường được đánh giá trung lập.


In [ ]:
counts = {aspect: absa_v1[aspect].value_counts() for aspect in aspects}
df_counts = pd.DataFrame(counts).fillna(0)
df_counts = df_counts.T  

df_counts.plot(kind="bar", figsize=(10, 6))
plt.title("Sentiment Distribution per Aspect")
plt.xlabel("Aspect")
plt.ylabel("Count")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

Biểu đồ thể hiện rõ xu hướng cảm xúc theo từng khía cạnh. Các cột trung lập chiếm ưu thế ở hầu hết các khía cạnh, đặc biệt là packaging, service và price. Chất lượng là khía cạnh có nhiều đánh giá tích cực nhất, cho thấy người dùng hài lòng với sản phẩm. Cảm xúc tiêu cực xuất hiện ít, tập trung ở mức thấp trên tất cả các khía cạnh. Tổng thể, dữ liệu phản ánh trải nghiệm người dùng chủ yếu trung lập hoặc tích cực, rất ít bất mãn rõ rệt.


In [ ]:
absa_v1.head()

## Version 2

In [ ]:
absa_v2 = pd.read_json('../data/absa/v2/full_dataset.json')

In [ ]:
absa_v2.shape

In [ ]:
absa_v2.info()

In [ ]:
absa_v2.isnull().sum()

Dữ liệu không có giá trị thiếu.

In [ ]:
absa_v2["review_length"] = absa_v2["text"].astype(str).apply(lambda x: len(x.split()))
absa_v2["review_length"].describe()

Dữ liệu cho thấy độ dài trung bình của đánh giá là khoảng 17 từ, với độ lệch chuẩn gần 20, phản ánh sự chênh lệch khá lớn giữa các bài viết. Phần lớn đánh giá ngắn, tập trung quanh giá trị trung vị 11 từ, trong khi một số ít đánh giá rất dài (tối đa 334 từ). Phân bố lệch phải rõ rệt, cho thấy người dùng thường viết ngắn gọn, chỉ một số ít cung cấp nhận xét chi tiết hơn.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(absa_v2["review_length"], bins=50)
plt.title("Review Length Distribution")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()

Biểu đồ cho thấy phần lớn đánh giá có độ dài rất ngắn, tập trung trong khoảng 0–10 từ. Tần suất giảm nhanh khi số từ tăng, và chỉ có một số ít đánh giá vượt quá 100 từ. Phân bố lệch phải rõ rệt, phản ánh xu hướng người dùng viết đánh giá ngắn gọn, hiếm khi viết dài. Điều này phù hợp với thống kê mô tả trước đó, cho thấy phần lớn nội dung đánh giá mang tính khái quát hơn là chi tiết.


In [ ]:
aspects = [
    "description",
    "quality",
    "packaging",
    "delivery",
    "service",
    "price"
]

for aspect in aspects:
    print(absa_v2[aspect].value_counts())

Description: phần lớn trung lập, ít cảm xúc rõ rệt.  
Quality: chủ yếu tích cực, thể hiện sự hài lòng về chất lượng sản phẩm.  
Packaging: đa số trung lập, bao bì không phải yếu tố nổi bật.  
Delivery: nhiều đánh giá tích cực, dịch vụ giao hàng được đánh giá tốt.  
Service: phần lớn trung lập, trải nghiệm dịch vụ không gây ấn tượng mạnh.  
Price: chủ yếu trung lập, giá cả không phải yếu tố chính trong đánh giá.

Tổng quan: người dùng có xu hướng đánh giá tích cực về chất lượng và giao hàng, trong khi các khía cạnh khác thường được nhận xét trung lập.


In [ ]:
counts = {aspect: absa_v2[aspect].value_counts() for aspect in aspects}
df_counts = pd.DataFrame(counts).fillna(0)
df_counts = df_counts.T  

df_counts.plot(kind="bar", figsize=(10, 6))
plt.title("Sentiment Distribution per Aspect")
plt.xlabel("Aspect")
plt.ylabel("Count")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

Biểu đồ thể hiện rõ sự khác biệt về cảm xúc giữa các khía cạnh. Các cột trung lập chiếm ưu thế ở hầu hết các khía cạnh, đặc biệt là packaging, service và price. Chất lượng là khía cạnh có nhiều đánh giá tích cực nhất, cho thấy người dùng hài lòng với sản phẩm. Cảm xúc tiêu cực xuất hiện ít, tập trung ở mức thấp trên tất cả các khía cạnh. Tổng thể, dữ liệu phản ánh trải nghiệm người dùng chủ yếu trung lập hoặc tích cực, rất ít bất mãn rõ rệt.


In [ ]:
absa_v2.head()